# ผู้ช่วยเลือกเมนูจากวัตถุดิบที่มีด้วย RAG

ชื่อผู้จัดทำ: [กรอกชื่อ]  รหัสนักศึกษา: [กรอก]  รายวิชา: [กรอก]

Streamlit URL: [ยังไม่ได้ Deploy — กรอกหลังใช้งานจริง]

GitHub URL: [ยังไม่ได้ Push — กรอกหลังสร้าง repository]

## แนวคิดและระบบ
อ่าน recipes.md → ##/### → chunks + chunk_id → Sentence Embeddings/FAISS → กลุ่มค้นกว้าง 60 chunks → กรองและจัดอันดับสูตร → เลือก Top-K chunks แบบกระจายสูตร → เติมหัวข้อสูตรเต็มแยกส่วน → Prompt จริง → Groq → ตรวจ recipe_ids/citations → แสดงคำตอบตรงประเด็นพร้อมหลักฐาน
K 1–10 ค่าเริ่มต้น 3 ควบคุม chunks ไม่ใช่การ์ด (ไม่เกิน 3) ไม่มี cache คำตอบ; cache index ผูกเนื้อหา/implementation/model ผู้ใช้เปลี่ยน K ได้โดยหลักฐานคำตอบก่อนหน้ายังเหมือนเดิม
LLM เลือกสูตรจาก context จริง ไม่แต่งสูตร แอปแสดงปริมาณ/ขั้นตอนจากข้อมูลต้นฉบับ สูตรทั้งหมดเป็นตัวอย่าง AI ต้องตรวจทานก่อนใช้จริง

## การติดตั้งบน Colab เพื่อทดสอบ Retrieval
กรอก GitHub URL ของคุณก่อนรัน ไม่ต้องใส่ API Key เพื่อทดสอบ Retrieval Colab ใช้สำหรับอธิบายและทดสอบโมดูล ไม่ใช่โฮสต์เว็บ Streamlit ถาวร

In [ ]:
REPOSITORY_URL = "YOUR_GITHUB_REPOSITORY_URL"
assert REPOSITORY_URL != "YOUR_GITHUB_REPOSITORY_URL", "กรอก URL GitHub ที่สร้างจริงก่อน"
import subprocess
subprocess.run(["git", "clone", REPOSITORY_URL, "menu-rag"], check=True)


In [ ]:
%cd /content/menu-rag
%pip install -r requirements-dev.txt


In [ ]:
from sentence_transformers import SentenceTransformer
from rag import MODEL
SentenceTransformer(MODEL, device="cpu", cache_folder=".cache/models")


In [ ]:
!python scripts/validate_data.py
!python -m pytest -q
!python scripts/evaluate.py


## ขั้นตอน 4: เขียน RAG Prompt Template
ใช้ build_rag_messages ซึ่ง select_with_llm ของแอปเรียกจริง แยก question, available_ingredients, retrieved_chunks และ CONTEXT สูตรเต็มเพิ่มเติม ข้อความสูตรเป็น reference data ไม่ใช่คำสั่ง ห้ามแต่งปริมาณ เวลา โภชนาการ ราคา วัตถุดิบ อุปกรณ์ หรือทดแทน คืน recipe_ids/citations ที่ตรวจสอบได้
โค้ดต่อไปสร้าง prompt จริงโดยยังไม่เรียก API ไม่มีผลลัพธ์จำลองแนบไว้

In [ ]:
import json
from rag import Retriever, load_recipes, candidates, build_rag_messages
embedding = SentenceTransformer(MODEL, device="cpu", cache_folder=".cache/models")
retriever = Retriever(load_recipes("data"), embedding)
question = "มีไข่ ข้าวสวย และต้นหอม ทำอะไรได้บ้าง"
trace = {}
hits = candidates(retriever, question, {"ไข่ไก่", "ข้าวสวย", "ต้นหอม"}, set(), top_k=3, trace=trace)
messages = build_rag_messages(question, hits)
print(json.dumps(messages, ensure_ascii=False, indent=2))


## การทดสอบ LLM (ทางเลือก)
ไม่บันทึก API Key ลง Notebook ใช้ Colab Secrets ชื่อ GROQ_API_KEY แล้วเรียก Groq โดยตรงตามตัวอย่างนี้ ยังไม่มีผลทดสอบ LLM จริงใน Notebook นี้

In [ ]:
from google.colab import userdata
from groq import Groq
from rag import Retriever, load_recipes, candidates, select_with_llm, render_answer
model = SentenceTransformer(MODEL, device="cpu", cache_folder=".cache/models")
retriever = Retriever(load_recipes("data"), model)
hits = candidates(retriever, "ข้าวผัดไข่ทำอย่างไร", {"ไข่ไก่", "ข้าวสวย", "ต้นหอม"}, set())
client = Groq(api_key=userdata.get("GROQ_API_KEY"), timeout=30, max_retries=0)
selected = select_with_llm(client, "openai/gpt-oss-120b", "ข้าวผัดไข่ทำอย่างไร", hits)
print(render_answer(selected))


## ผลทดสอบและข้อจำกัด
ดู TEST_REPORT.md และ retrieval_results.json สำหรับผลที่รันจริง ตรวจผล LLM กับ expected_answer ใน test_questions.csv โดยมนุษย์ด้วย ไม่ใช้ LLM-as-Judge เพียงอย่างเดียว

การตรวจของที่มีใช้ชื่อและ alias ที่กำหนด ไม่รับประกันปริมาณเพียงพอ การอ่านภาษาไทยแบบอิสระยังมีข้อจำกัด ไม่มีข้อมูลโภชนาการ/เวลาเป็นตัวเลข จึงปฏิเสธคำถามดังกล่าว ไม่มี Key จะเปิดเว็บได้แต่ไม่สร้างคำตอบปลอม

## ภาพหน้าจอหลังเว็บใช้งานจริง
1. หน้าแรกและข้อมูลคลังสูตร
2. คำตอบเมื่อวัตถุดิบครบ
3. คำตอบที่ขาดเครื่องปรุง
4. เงื่อนไขไมโครเวฟ
5. ถามต่อเมนูที่สอง
6. หลักฐานอ้างอิง
7. ปฏิเสธคำถามโภชนาการ
8. หน้าตั้งค่า Secrets โดยปิดบังคีย์ทั้งหมด

[เพิ่มภาพ Capture จริงภายหลัง — ไม่มีภาพจำลอง]

## สถานะคำถาม ตัวอย่าง และผลตรวจรอบล่าสุด
ช่องกรอกเป็นของที่บันทึก ไม่ถูกเขียนทับจากแชต วัตถุดิบในคำถามใหม่แทนชุดค้นเดิม คำว่า “เพิ่ม” รวมกับของที่บันทึก/บริบทล่าสุด; ถาม “เมนูนี้ใช้ไข่กี่ฟอง” รักษาเมนูที่เลือกและรายการเดิม เริ่มใหม่ล้างบทสนทนาและของในครัว แต่คง Top-K
ตัวอย่างตอบได้: วัตถุดิบไข่/ข้าว/ต้นหอม (R01/R05/R02), ปริมาณไข่ตุ๋น (R04, วัตถุดิบ/จำนวนเสิร์ฟ), วิธีทำข้าวผัด (R01, ขั้นตอน), อุปกรณ์ไข่ตุ๋น (R04, อุปกรณ์) ตัวอย่างปฏิเสธ: แคลอรี่ ราคา พิซซ่า และมาม่า/บะหมี่กึ่งสำเร็จรูปที่ไม่มีทั้งคลัง ดู supporting_sections ใน test_questions.csv
ผลทดสอบจริงดู STATE_TOPK_REPORT.md และ retrieval_results.json Tests แบบ mock ตรวจ interaction/error แยกจาก live Groq ไม่รัน Notebook นี้ทั้งเล่มเพราะต้องตั้ง repository และ Colab Secrets ไม่มีผลลัพธ์ที่แต่งขึ้นหรือ API Key ใน Notebook ยังไม่ได้ตรวจภาพ desktop/mobile จริงหรือ Deploy